In [1]:
!pip install -r "C:/Users/Lenovo/Desktop/Lorenzo/Code/Training/notebooks/requirements_py314.txt"

Note: you may need to restart the kernel to use updated packages.


# DL Evaluation Notebook (Python 3.14 - CPU)

This is the **CPU-only version** of the deep learning model evaluation notebook.

**For GPU/CUDA acceleration, use `dl_eval_notebook_cuda.ipynb` instead.**

This notebook runs the deep learning model evaluation workflow from `dl_eval.py`.
It performs data loading, model evaluation across multiple splits, generates predictions, and produces evaluation reports including confusion matrices, classification reports, and Grad-CAM visualizations.

**Device:** CPU (PyTorch will use CPU for all operations)

## Section 1: Import Required Libraries

Import all necessary libraries and modules required by the evaluation workflow.

In [2]:
# Import Required Libraries
from dl_funcs_notebook_cpu import (
    eval_process_folders, eval_create_dataloaders, eval_get_evals, eval_grad_cam
)
import torch
import os
import warnings
import numpy as np
import matplotlib
import pandas as pd
from datetime import datetime, timedelta
from ast import literal_eval

# Configure matplotlib and suppress warnings
matplotlib.use('agg')  # Use non-interactive backend
warnings.filterwarnings('ignore')

# Check device availability (CPU-only for this notebook)
device = torch.device("cpu")  # Force CPU for this notebook
print("="*80)
print("DEVICE SETUP (CPU-ONLY)")
print("="*80)
print(f"Device Available: {device}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("Note: CUDA is available but this notebook uses CPU.")
    print("For GPU acceleration, use dl_eval_notebook_cuda.ipynb")
print("="*80)

DEVICE SETUP (CPU-ONLY)
Device Available: cpu
CUDA Available: False


## Section 2: Configuration and Setup

Define paths, parameters, and reference mappings for the evaluation workflow.

In [3]:
# Configuration and Setup

# Define paths
reference_path = "C:/Users/Lenovo/Desktop/Lorenzo/Test/reference_list.txt"
source_dir = "C:/Users/Lenovo/Desktop/Lorenzo/Test/Fixed_Preserved"
export_dir = "C:/Users/Lenovo/Desktop/Lorenzo/Test/res"
model_src = "C:/Users/Lenovo/Desktop/Lorenzo/Code/Models/Fixed/models"
index_file = "C:/Users/Lenovo/Desktop/Lorenzo/Test/Fixed_feat.csv"
data_pkl_path = "C:/Users/Lenovo/Desktop/Lorenzo/Test/data_dl.pkl"
output_base_dir = "C:/Users/Lenovo/Desktop/Lorenzo/Test/res"

# Parameters
num_samples = 250
watch_list = ['Fixed_Preserved', 'Others']
watch_list.sort()

# CPU batch size (smaller for CPU processing)
batch_size = 16  # Smaller batch for CPU
print(f"Batch size: {batch_size} (CPU-optimized)")

# Load reference mapping
reference = {}
with open(reference_path, "r") as f:
    file_contents = f.readlines()
    for line in file_contents:
        temp = line.split(":", 2)
        if len(temp) == 2:
            reference[str(temp[1]).strip()] = int(temp[0])

watch_list = list(reference.keys())
print(f"Reference mapping: {reference}")
print(f"Watch list (classes): {watch_list}")

# Helper function to prepare output folders
def prepareOutputFolder(exp_dir):
    if not os.path.exists(exp_dir):
        os.makedirs(exp_dir, exist_ok=True)

print("Configuration loaded successfully!")

Batch size: 16 (CPU-optimized)
Reference mapping: {'Chaetoceros': 0, 'Others': 1}
Watch list (classes): ['Chaetoceros', 'Others']
Configuration loaded successfully!


## Section 3: Data Loading

Load or generate the dataset from the source folder. This step extracts images and processes them into a DataFrame.

In [4]:
# Data Loading

data = pd.DataFrame()

if not os.path.exists(data_pkl_path):
    print("No saved copy of dataset found, generating from source folder...")
    data = eval_process_folders(source_dir, reference)
else:
    print("A saved copy of the dataset was found, loading...")
    data = pd.read_pickle(data_pkl_path)

print(f"Dataset loaded. Total records: {len(data)}")
print(f"Dataset shape: {data.shape}")
print(f"Dataset columns: {data.columns.tolist()}")
print(f"\nFirst few rows:")
print(data.head())

A saved copy of the dataset was found, loading...
Dataset loaded. Total records: 70
Dataset shape: (70, 8)
Dataset columns: ['image', 'filename', 'roi_number', 'label', 'volume_analyzed', 'humidity', 'temperature', 'image_size']

First few rows:
                                               image  \
0  [[0.8196078431372549, 0.8196078431372549, 0.80...   
1  [[0.8117647058823529, 0.8235294117647058, 0.8,...   
2  [[0.792156862745098, 0.7843137254901961, 0.8, ...   
3  [[0.792156862745098, 0.7725490196078432, 0.788...   
4  [[0.7764705882352941, 0.7843137254901961, 0.80...   

                             filename  roi_number  label  volume_analyzed  \
0  D20250609T054352_IFCB202_10_10.png          -1      1               -1   
1  D20250609T054352_IFCB202_11_11.png          -1      1               -1   
2  D20250609T054352_IFCB202_12_12.png          -1      1               -1   
3  D20250609T054352_IFCB202_13_13.png          -1      1               -1   
4  D20250609T054352_IFCB202_14_1

## Section 4: Evaluation Execution (Split Loop)

In [5]:
# Initialize results tracking
results_summary = {}
all_predictions = {}
all_ground_truth = {}

# Run evaluation for splits 1-5
for split_num in range(1, 6):
    print(f"\n{'='*80}")
    print(f"SPLIT {split_num}")
    print(f"{'='*80}")
    
    try:
        # Create dataloaders
        print(f"Creating dataloaders for split {split_num}...")
        train_loader, val_loader, test_loader = eval_create_dataloaders(
            data, split_num, batch_size=batch_size
        )
        
        # Run evaluation
        print(f"Running model evaluation for split {split_num}...")
        val_metrics, test_metrics, all_preds, all_true = eval_get_evals(
            split_num, val_loader, test_loader
        )
        
        # Store results
        results_summary[f'split_{split_num}'] = {
            'val_metrics': val_metrics,
            'test_metrics': test_metrics
        }
        all_predictions[f'split_{split_num}'] = all_preds
        all_ground_truth[f'split_{split_num}'] = all_true
        
        # Export results to CSV
        results_dir = os.path.join(output_base_dir, f'split_{split_num}')
        os.makedirs(results_dir, exist_ok=True)
        
        # Save evaluation metrics
        metrics_df = pd.DataFrame({
            'Metric': list(test_metrics.keys()),
            'Test_Value': list(test_metrics.values())
        })
        metrics_df.to_csv(os.path.join(results_dir, f'eval_metrics_split_{split_num}.csv'), index=False)
        
        # Save predictions
        preds_df = pd.DataFrame({
            'Predictions': all_preds,
            'Ground_Truth': all_true
        })
        preds_df.to_csv(os.path.join(results_dir, f'predictions_split_{split_num}.csv'), index=False)
        
        print(f"Results exported to {results_dir}")
        print(f"Test Metrics: {test_metrics}")
        
    except Exception as e:
        print(f"Error processing split {split_num}: {str(e)}")

print(f"\n{'='*80}")
print("All splits evaluated successfully!")
print(f"{'='*80}")


SPLIT 1
Creating dataloaders for split 1...
Running model evaluation for split 1...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_1
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}

SPLIT 2
Creating dataloaders for split 2...
Running model evaluation for split 2...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_2
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}

SPLIT 3
Creating dataloaders for split 3...
Running model evaluation for split 3...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_3
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}

SPLIT 4
Creating dataloaders for split 4...
Running model evaluation for split 4...
Results exported to C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_4
Test Metrics: {'accuracy': 0.82, 'precision': 0.81, 'recall': 0.82}

SPLIT 5
Creating dataloaders for split 5...
Running model evaluation for split 5...
Results exported to C:/User

## Section 5: Grad-CAM Visualization and Results

In [6]:
# Generate Grad-CAM visualizations for each split
for split_num in range(1, 6):
    print(f"\nGenerating Grad-CAM visualizations for split {split_num}...")
    
    try:
        # Create dataloaders again for Grad-CAM
        train_loader, val_loader, test_loader = eval_create_dataloaders(
            data, split_num, batch_size=batch_size
        )
        
        # Run Grad-CAM
        eval_grad_cam(
            split_num, test_loader,
            reference=reference,
            device=device,
            export_tree=output_base_dir,
            model_source=model_src,
            num_samples=num_samples
        )
        
        print(f"Grad-CAM visualizations saved for split {split_num}")
        
    except Exception as e:
        print(f"Error generating Grad-CAM for split {split_num}: {str(e)}")
        continue

print("\nGrad-CAM generation complete for all splits!")


Generating Grad-CAM visualizations for split 1...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_1\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:16<00:00,  1.98s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:25<00:00,  2.05s/it]


Grad-CAM visualizations saved for split 1

Generating Grad-CAM visualizations for split 2...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_2\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:17<00:00,  1.73s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:29<00:00,  2.07s/it]


Grad-CAM visualizations saved for split 2

Generating Grad-CAM visualizations for split 3...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_3\grad_cam


...

Grad-CAM visualizations saved for split 3

Generating Grad-CAM visualizations for split 4...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_4\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:08<00:00,  1.84s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:14<00:00,  1.95s/it]


Grad-CAM visualizations saved for split 4

Generating Grad-CAM visualizations for split 5...
C:/Users/Lenovo/Desktop/Lorenzo/Test/res\split_5\grad_cam


0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:15<00:00,  1.99s/it]
0it [00:00, ?it/s]
100%|██████████████████████████████████████████████████████████████████████████████████| 70/70 [03:29<00:00,  1.97s/it]


Grad-CAM visualizations saved for split 5

Grad-CAM generation complete for all splits!


## Section 6: Results Summary

In [7]:
# Display summary of results
print("\n" + "="*80)
print("EVALUATION SUMMARY ACROSS ALL SPLITS")
print("="*80)

# Aggregate metrics across splits
aggregated_metrics = {}
for split_key, metrics in results_summary.items():
    split_num = split_key.split('_')[1]
    test_metrics = metrics['test_metrics']
    print(f"\nSplit {split_num} Test Metrics:")
    for metric_name, metric_value in test_metrics.items():
        print(f"  {metric_name}: {metric_value:.4f}")
        if metric_name not in aggregated_metrics:
            aggregated_metrics[metric_name] = []
        aggregated_metrics[metric_name].append(metric_value)

# Calculate mean and std for each metric
print(f"\n{'='*80}")
print("CROSS-SPLIT STATISTICS")
print(f"{'='*80}")
for metric_name, values in aggregated_metrics.items():
    mean_val = np.mean(values)
    std_val = np.std(values)
    print(f"{metric_name}:")
    print(f"  Mean: {mean_val:.4f}")
    print(f"  Std:  {std_val:.4f}")
    print(f"  Min:  {min(values):.4f}")
    print(f"  Max:  {max(values):.4f}")

print(f"\n{'='*80}")
print("Evaluation and visualization complete!")
print(f"Results saved to: {output_base_dir}")
print(f"{'='*80}")


EVALUATION SUMMARY ACROSS ALL SPLITS

Split 1 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 2 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 3 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 4 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

Split 5 Test Metrics:
  accuracy: 0.8200
  precision: 0.8100
  recall: 0.8200

CROSS-SPLIT STATISTICS
accuracy:
  Mean: 0.8200
  Std:  0.0000
  Min:  0.8200
  Max:  0.8200
precision:
  Mean: 0.8100
  Std:  0.0000
  Min:  0.8100
  Max:  0.8100
recall:
  Mean: 0.8200
  Std:  0.0000
  Min:  0.8200
  Max:  0.8200

Evaluation and visualization complete!
Results saved to: C:/Users/Lenovo/Desktop/Lorenzo/Test/res
